Downloading and verifying the Dataset:

In [ ]:
import kagglehub
kagglehub.dataset_download('sadmanhsakib/hc18-preprocessed-dataset')

import os
print(os.listdir("/kaggle/input"))

Dynamic Data mapping for the data.yml file:

In [ ]:
import yaml
from pathlib import Path

# Path where Kaggle mounts your dataset
KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/sadmanhsakib/hc18-preprocessed-dataset/yolo")

# Read local data.yaml
with open(KAGGLE_DATASET_DIR / "data.yaml", "r") as f:
    config = yaml.safe_load(f)

# Patch the path to the Kaggle input location
config["path"] = str(KAGGLE_DATASET_DIR.resolve())

# Write patched config to Kaggle working directory
with open("/kaggle/working/data_patched.yaml", "w") as f:
    yaml.safe_dump(config, f)

print("Patched data.yaml written to /kaggle/working/data_patched.yaml")

Installing Dependancies:

In [ ]:
!pip install ultralytics

Training the the segmentation model:

In [ ]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLOv8s segmentation model
model = YOLO("yolov8s-seg.pt")

# Train the model
results = model.train(
    data="/kaggle/working/data_patched.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,               # Use GPU
    optimizer="AdamW",
    lr0=0.01,
    patience=10,            # Early stopping after 10 epochs without improvement
    project="fetalmetrics-ai",
    name="yolov8s_seg_baseline",
    val=True,
    plots=True
)

Dice Coefficient Validation:

In [24]:
import numpy as np
import cv2
from pathlib import Path
from ultralytics import YOLO

def calculate_dice(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """Compute Dice Coefficient between two binary masks."""
    intersection = np.sum((pred_mask > 0) & (gt_mask > 0))
    total_pixels = np.sum(pred_mask > 0) + np.sum(gt_mask > 0)
    if total_pixels == 0:
        return 1.0 if np.sum(gt_mask) == 0 else 0.0
    return (2.0 * intersection) / total_pixels

# Load the trained baseline weights
model = YOLO("/kaggle/working/runs/segment/fetalmetrics-ai/yolov8s_seg_baseline/weights/best.pt")

# Predict on validation images and compute average Dice
val_images_dir = Path("/kaggle/input/datasets/sadmanhsakib/hc18-preprocessed-dataset/yolo/images/val")
val_labels_dir = Path("/kaggle/input/datasets/sadmanhsakib/hc18-preprocessed-dataset/yolo/labels/val")

dice_scores = []

for img_path in val_images_dir.glob("*.png"):
    # Load corresponding ground truth label (FastAI mask or reconstruct from YOLO polygon)
    # For convenience, load from fastai/masks/val directory if mounted:
    mask_path = Path(str(img_path).replace("yolo/images", "fastai/masks"))
    gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    # Run prediction
    results = model.predict(img_path, imgsz=640, verbose=False)

    # Extract predicted mask
    pred_mask = np.zeros_like(gt_mask)
    if results[0].masks is not None:
        # Resize predicted mask back to original image shape
        masks_data = results[0].masks.data.cpu().numpy()[0]
        pred_mask = cv2.resize(masks_data, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)
        pred_mask = (pred_mask > 0.5).astype(np.uint8) * 255

    dice = calculate_dice(pred_mask, gt_mask)
    dice_scores.append(dice)

mean_dice = np.mean(dice_scores)
print(f"Validation Mean Dice Coefficient: {mean_dice:.4f}")

Validation Mean Dice Coefficient: 0.9615


Model Optimization:

In [ ]:
results_optimized = model.train(
    data="/kaggle/working/data_patched.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.005,              # Try slightly smaller learning rate for fine tuning
    patience=12,
    # Augmentation Overrides
    flipud=0.0,             # Disable vertical flips (Clinical Safety)
    fliplr=0.5,             # Enable horizontal flips
    degrees=15.0,           # Rotate up to 15 degrees
    scale=0.15,             # Scale scaling range to 85%-115%
    hsv_h=0.0,              # Disable hue augmentations (Grayscale data)
    hsv_s=0.0,              # Disable saturation augmentations
    hsv_v=0.15,             # Keep brightness adjustments (v = value)
    project="fetalmetrics-ai",
    name="yolov8s_seg_optimized",
    val=True,
    plots=True
)

Tuning learning rate and weight decay:


In [ ]:
model.tune(
    data="/kaggle/working/data_patched.yaml",
    epochs=30,
    iterations=10,          # Runs 10 iterations of genetic search
    optimizer="AdamW",
    plots=False,
    save=False,
    val=True
)

Tuner: Initialized Tuner instance with 'tune_dir=/kaggle/working/runs/segment/tune'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
WARNING ⚠️ 'model' argument is missing. Using default 'model=yolo26n.pt'.
WARNING ⚠️ conflicting 'task=segment' passed with 'task=detect' model. Ignoring 'task=segment' and updating to 'task=detect' to match model.
Ultralytics 8.4.81 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, a